# Riesgo Crediticio, Clustering y Explicabilidad (XAI)
### Pipeline de Machine Learning Libre de Data Leakage y Auditoría SHAP

**Autor:** Luis Monge  
**Metodología:** CRISP-DM  
**Dataset Oficial:** Online Shoppers Purchasing Intention (UCI ML Repository)

## 1. Carga Adaptativa de Datos e Ingesta

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_loader import load_dataset
from src.feature_pipeline import construir_preprocesador_shoppers
from src.clustering import proyectar_pca_2d, entrenar_modelos_segmentacion, perfilar_segmentos_negocio
from src.supervised_xai import entrenar_evaluar_logistica, entrenar_evaluar_random_forest, calcular_explicabilidad_shap

df = load_dataset(force_sample=False)
print(f"Registros cargados: {len(df)}")
print(df.head(2))

## 2. Pipeline Scikit-Learn Libre de Data Leakage

In [2]:
X = df.drop(columns=['Revenue'])
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
preproc = construir_preprocesador_shoppers()

X_train_p = preproc.fit_transform(X_train)
X_test_p = preproc.transform(X_test)
print(f"Dimensión X_train procesada: {X_train_p.shape} | Cero NaNs: {not np.isnan(X_train_p).any()}")

## 3. Segmentación No Supervisada & Arquetipos de Negocio

In [3]:
coords_pca, pca_obj = proyectar_pca_2d(X_train_p, n_componentes=2)
etiquetas = entrenar_modelos_segmentacion(X_train_p, k=3)
resumen = perfilar_segmentos_negocio(X_train.assign(Revenue=y_train), etiquetas['kmeans'])
print("Arquetipos de Comportamiento:")
print(resumen)

## 4. Benchmark Supervisado & Explicabilidad SHAP

In [4]:
clf_lr, met_lr = entrenar_evaluar_logistica(X_train_p, y_train, X_test_p, y_test)
clf_rf, met_rf = entrenar_evaluar_random_forest(X_train_p, y_train, X_test_p, y_test)

comp_df = pd.DataFrame([met_lr, met_rf], index=['Regresión Logística L2', 'Random Forest'])
print(comp_df)

shap_res = calcular_explicabilidad_shap(clf_rf, X_test_p[:50])
print("Top 5 Variables de Mayor Impacto:")
for var, val in shap_res['ranking_variables'][:5]:
    print(f"- {var}: {val:.4f}")